# 🎬 Movie Recommendation System
**Project 3 — AI Recommendation Logic | DecodeLabs 2026**

A content-based filtering engine that recommends movies similar to a user's input,
using TF-IDF vectorization and Cosine Similarity.

**Pipeline:**
1. Load & inspect data
2. Parse and engineer features (genres + keywords + overview)
3. Build TF-IDF matrix
4. Compute Cosine Similarity
5. Return Top-N recommendations

In [21]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
df = pd.read_csv("tmdb_5000_movies.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print(df.head(3))
print("Info:\n", df.info())
print("Desc:\n", df.describe())

Shape: (4803, 20)

Columns: ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']
      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   19995   
1  http://disney.go.com/disneypictures/pirates/     285   
2   http://www.sonypictures.com/movies/spectre/  206647   

                                            keywords original_language  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...                en   
1  [{"id": 270, "name": "ocean"}, {"

## Feature Engineering

The dataset stores `genres` and `keywords` as stringified JSON.
We need to parse them into plain text so TF-IDF can process them.

We combine three features per movie:
- **genres** — broad category (Action, Drama, etc.)
- **keywords** — specific themes (based on a true story, revenge, etc.)
- **overview** — plain text plot summary

Together these form one "feature string" per movie that represents its content.

In [22]:
def parse_json_column(text):
    """Converts stringified JSON like genres/keywords into a list of names."""
    try:
        items = ast.literal_eval(text)   #from text strings -> python list of dictionaries.
        return [item['name'].lower().replace(" ", "_") for item in items]
    except:
        return []

df['genres_parsed']   = df['genres'].apply(parse_json_column)
df['keywords_parsed'] = df['keywords'].apply(parse_json_column)

'''
Before:
"[{"id": 28, "name": "Action"}, {"id": 53, "name": "Thriller"}]"
After:
python['action', 'thriller']
'''
# Verify it worked
print(df['genres_parsed'].iloc[0])
print(df['keywords_parsed'].iloc[0])

['action', 'adventure', 'fantasy', 'science_fiction']
['culture_clash', 'future', 'space_war', 'space_colony', 'society', 'space_travel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alien_planet', 'cgi', 'marine', 'soldier', 'battle', 'love_affair', 'anti_war', 'power_relations', 'mind_and_soul', '3d']


In [23]:
'''
TF-IDF is basically a word counter. It can only count words from a single piece of text. It cannot read three separate lists. So we need to squish everything into:
"action thriller joker gotham a dark hero fights crime"
'''
def build_feature_string(row):
    genres   = " ".join(row['genres_parsed'])
    keywords = " ".join(row['keywords_parsed'])
    overview = str(row['overview']).lower() if pd.notna(row['overview']) else ""
    return f"{genres} {keywords} {overview}"    #glues all three together into one string.

df['features'] = df.apply(build_feature_string, axis=1)

df = df[['title', 'features']].dropna().reset_index(drop=True)

print(f"Clean dataset: {df.shape[0]} movies")
df.head(5)

Clean dataset: 4803 movies


,title,features
0,Avatar,action adventure fantasy science_fiction cultu...
1,Pirates of the Caribbean: At World's End,adventure fantasy action ocean drug_abuse exot...
2,Spectre,action adventure crime spy based_on_novel secr...
3,The Dark Knight Rises,action crime drama thriller dc_comics crime_fi...
4,John Carter,action adventure science_fiction based_on_nove...


## TF-IDF Vectorization

**Why TF-IDF over simple binary (0/1) vectors?**

Binary vectors treat every word equally.
TF-IDF assigns higher weight to words that are:
- Frequent within a specific movie (Term Frequency)
- Rare across the entire dataset (Inverse Document Frequency)

This means a niche keyword like "dystopian" matters more than a generic word like "story".

`max_features=5000` keeps the vocabulary manageable without losing important terms.
`stop_words='english'` removes words like "the", "and", "is" that carry no meaning.

In [24]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = vectorizer.fit_transform(df['features'])

print("TF - IDF matrix shape:", tfidf_matrix.shape)

TF - IDF matrix shape: (4803, 5000)


## Cosine Similarity

**Why Cosine Similarity over Euclidean distance?**

Euclidean distance measures the straight-line gap between two points.
It is sensitive to vector magnitude — a movie with a long overview would
appear "far" from a movie with a short one, even if they share the same themes.

Cosine Similarity measures the **angle** between two vectors, not their length.
It is magnitude-invariant, making it the industry standard for text similarity.

- Score **1.0** = identical orientation (very similar content)
- Score **0.0** = orthogonal (no shared features)

In [25]:
def get_recommendations(movie_title, top_n=5):

    # Find movie index (case-insensitive)
    matches = df[df['title'].str.lower() == movie_title.lower()]

    if matches.empty:
        # Try partial match
        matches = df[df['title'].str.lower().str.contains(movie_title.lower())]   #If exact match fails, tries a partial match. So typing "dark knight" would still find "The Dark Knight"

    if matches.empty:
        print(f"Movie '{movie_title}' not found in dataset.")
        return None

    movie_idx = matches.index[0]
    matched_title = df.loc[movie_idx, 'title']
    print(f"Found: '{matched_title}' (index {movie_idx})")

    # Get this movie's vector and compute similarity against all others
    movie_vector = tfidf_matrix[movie_idx]
    similarity_scores = cosine_similarity(movie_vector, tfidf_matrix)[0]

    # Build a series, drop the input movie itself
    score_series = pd.Series(similarity_scores, index=df.index)
    score_series = score_series.drop(index=movie_idx)

    # Sort descending, take top N
    top_matches = score_series.sort_values(ascending=False).head(top_n)

    # Build results dataframe
    results = pd.DataFrame({
        'rank': range(1, top_n + 1),
        'title': df.loc[top_matches.index, 'title'].values,
        'similarity_score': top_matches.values.round(4)
    })

    return results

In [26]:
# Change this to any movie in the dataset
user_input = "the Fast and the Furious"

recommendations = get_recommendations(user_input, top_n=5)

if recommendations is not None:
    print(f"\nTop 5 movies similar to '{user_input}':\n")
    print(recommendations.to_string(index=False))

Found: 'The Fast and the Furious' (index 99)

Top 5 movies similar to 'the Fast and the Furious':

 rank                                 title  similarity_score
    1                      2 Fast 2 Furious            0.2648
    2 The Fast and the Furious: Tokyo Drift            0.2466
    3                        Need for Speed            0.2225
    4                            Stone Cold            0.2142
    5                           Point Break            0.2039


In [27]:
for title in ["Avatar", "Ratatouille", "The Godfather"]:
    print(f"\n{'='*50}")
    print(f"Input: {title}")
    print('='*50)
    result = get_recommendations(title, top_n=3)
    if result is not None:
        print(result.to_string(index=False))


Input: Avatar
Found: 'Avatar' (index 0)
 rank         title  similarity_score
    1 Falcon Rising            0.2162
    2     Apollo 18            0.1859
    3    Titan A.E.            0.1809

Input: Ratatouille
Found: 'Ratatouille' (index 118)
 rank               title  similarity_score
    1     No Reservations            0.2779
    2               Burnt            0.2433
    3 Simply Irresistible            0.2306

Input: The Godfather
Found: 'The Godfather' (index 3337)
 rank                   title  similarity_score
    1  The Godfather: Part II            0.1961
    2  The Master of Disguise            0.1824
    3 The Godfather: Part III            0.1816


In [28]:
import os

# Save the last recommendations to outputs/
os.makedirs("../outputs", exist_ok=True)

final_result = get_recommendations("Avatar", top_n=10)
if final_result is not None:
    final_result.to_csv("../outputs/recommendations.csv", index=False)
    print("Saved to outputs/recommendations.csv")

Found: 'Avatar' (index 0)
Saved to outputs/recommendations.csv
